In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [4]:
# A

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Tugas7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

USER = "mahasiswa"
RAW = f"hdfs://localhost:9000/user/{USER}/tugas7/raw"

df_transaksi = spark.read.csv(f"{RAW}/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang    = spark.read.json(f"{RAW}/tugas7_cabang.json")
df_menu      = spark.read.csv(f"{RAW}/tugas7_menu.csv", header=True, inferSchema=True)

print(df_transaksi.count(), df_cabang.count(), df_menu.count())

4000 3 15


In [5]:
# B

df_gabungan = (
    df_transaksi
    .join(df_cabang, on="cabang_id")
    .join(df_menu, on="menu_id")
    .withColumn("total_penjualan", col("qty") * col("harga"))
)
df_gabungan.show(5)
print("Total baris:", df_gabungan.count())   # harus 4000

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [6]:
# C

penjualan_menu = (
    df_gabungan
    .groupBy("nama_cabang", "nama_menu")
    .agg(spark_sum("total_penjualan").alias("omzet_menu"))
)

w = Window.partitionBy("nama_cabang").orderBy(col("omzet_menu").desc())

top2_menu = (
    penjualan_menu
    .withColumn("peringkat", row_number().over(w))
    .filter(col("peringkat") <= 2)
    .orderBy("nama_cabang", "peringkat")
)
top2_menu.show()

+--------------------+---------+----------+---------+
|         nama_cabang|nama_menu|omzet_menu|peringkat|
+--------------------+---------+----------+---------+
|Kopi Nusantara Ma...|   Menu-1|   8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|   8085000|        2|
|Kopi Nusantara Se...|  Menu-13|   8750000|        1|
|Kopi Nusantara Se...|   Menu-9|   8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|   9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|   8160000|        2|
+--------------------+---------+----------+---------+



In [7]:
# D

df_gabungan.createOrReplaceTempView("gabungan")

spark.sql("""
    SELECT kepala_barista,
           SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""").show()

[Stage 46:>                                                         (0 + 1) / 1]

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [8]:
# E

OUT = f"hdfs://localhost:9000/user/{USER}/tugas7/processed/laporan"

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(OUT)

df_cek = spark.read.parquet(OUT)
print("Jumlah baris terbaca ulang:", df_cek.count())   # harus 4000
df_cek.printSchema()

Jumlah baris terbaca ulang: 4000
root
 |-- menu_id: integer (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- total_penjualan: integer (nullable = true)
 |-- kategori_menu: string (nullable = true)



In [9]:
# F

ringkasan = (
    df_gabungan
    .groupBy("nama_cabang")
    .agg(
        spark_sum("total_penjualan").alias("total_penjualan"),
        count("trx_id").alias("jumlah_transaksi"),
        avg("total_penjualan").alias("rata2_nilai_transaksi")
    )
    .orderBy(col("total_penjualan").desc())
)
ringkasan.show(truncate=False)

+-------------------------+---------------+----------------+---------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata2_nilai_transaksi|
+-------------------------+---------------+----------------+---------------------+
|Kopi Nusantara Semarang  |90542000       |1393            |64997.8463747308     |
|Kopi Nusantara Magelang  |87138000       |1302            |66926.26728110599    |
|Kopi Nusantara Yogyakarta|84645000       |1305            |64862.06896551724    |
+-------------------------+---------------+----------------+---------------------+



Cabang yang berkinerja terbaik adalah **Kopi Nusantara Semarang**, sedangkan cabang dengan kinerja terlemah adalah **Kopi Nusantara Yogyakarta**. Berdasarkan hasil window function pada bagian C, dua menu terlaris di cabang Yogyakarta adalah **Menu-12** (omzet Rp9.275.000) dan **Menu-15** (omzet Rp8.160.000), sehingga kedua menu inilah yang sebaiknya dipromosikan lebih gencar di cabang tersebut, misalnya lewat paket bundling, diskon pada jam sibuk, atau penempatan menu unggulan di etalase dan aplikasi pemesanan.

Alasannya, kedua menu tersebut sudah terbukti paling diminati pelanggan di Yogyakarta, sehingga peluang konversinya paling besar dan hasilnya lebih cepat terlihat dibandingkan memperkenalkan menu baru dari nol. Promosi yang difokuskan pada menu andalan juga lebih efisien secara biaya, dan dapat membantu cabang Yogyakarta mengejar selisih omzet dengan Semarang. Strategi bundling dengan menu pendamping juga berpotensi menaikkan nilai rata-rata per transaksi, sehingga total penjualan cabang ikut meningkat.